# 06. JaxPIP 连接势能面与 JAX GPU 张量加速动力学案例

传统的单项式对称化方法（MSA）通过深度嵌套的标量因子分解计算置换不变多项式（PIP），在现代自动微分与 GPU 编译器中会产生庞大的计算图；而 **JaxPIP** 将 PIP / PIP-NN 重构为规整的稠密矩阵乘法、`log-exp` 变换与分段求和（Segmented Summation），天然适配 XLA 编译器的宽批量矢量化执行。

在路径积分分子动力学（PIMD / RPMD）中，$B$ 条独立轨迹 $\times$ $P$ 个环聚合物珠子每一步需要同时计算 $B \times P$ 个分子构型的能量与原子力。本教程展示 **JaxPIP + JAX GPU** 与 **PyRPMD** 连接的三种递进加速模式：

1. **标准 `Potential` 适配器 (`JaxPIPPotential`)**：基于 `@eqx.filter_jit` 与 `jax.vmap(model.get_energy_and_forces)` 封装束缚态水分子（$\mathrm{H_2O}$）PIP 与 PIP-NN 势能面，无缝接入 `check_potential`、`Dynamics`、`prepare` 等全部 PyRPMD 接口。
2. **DLPack 零拷贝桥接接入 `PyRPMD` 设备驻留张量引擎 (`JaxPIPTorchBridge` $\rightarrow$ `TorchPES` + `TorchDynamics` + `TorchSplitPotential`)**：利用标准 DLPack 协议在 GPU 显存（或主机内存）上**零拷贝共享数据指针**，直接驱动 `cayley-bcocb` + `pile-l` 采样以及**环聚合物收缩（RPC）+ 多时间步长（MTS）**（快变线性 PIP 在 $P=16$ 全珠子求值，慢变 PIP-NN 修正收缩至 $P_c=4$ 珠子求值）。
3. **纯 JAX XLA 全编译融合传播循环 (`jax.lax.scan` + `jax.vmap`)**：将法向模 Cayley 自由环演化、`rfft`/`irfft` 与 `JaxPIP` 批量力求值编译进单个 XLA 图内连续演化多步，消除逐步 Python 调度开销，并与 `PyRPMD` 的 `TorchDynamics` 验证双精度机器级逐帧一致性（误差 $< 10^{-12}$）。
4. **大规模环聚合物批量吞吐量扩展评测 (`B * P` Batch Scaling)**：评测单步并发构型数从 `16` 扩展至 `1024`（$B \in \{1, 8, 32, 64\}, P = 16$）时的单珠子摊销耗时与吞吐量。

> **硬件自适应说明**：当安装了 CUDA 版 JAX（如 `pip install "jax[cuda12]"`）与 CUDA 版 PyTorch 时，所有张量计算自动在 NVIDIA GPU 上以 `float64` 执行；在无 GPU 的机器或 CI 环境中自动使用 XLA CPU 后端运行同一套代码，并如实报告硬件后端。

In [ ]:
import json
import time
import numpy as np

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import equinox as eqx
import torch
from jaxpip.descriptor import PolynomialDescriptor
from jaxpip.model import PolynomialLinearModel, PolynomialNeuralNetwork

import pyrpmd
from pyrpmd.constants import AMU_TO_AU, BOHR_ANGSTROM, EV_TO_AU
from pyrpmd.potentials import PESResult, Potential, check_potential
from pyrpmd.torch_backend import (
    TorchDynamics,
    TorchPES,
    TorchRPCTerm,
    TorchSplitPotential,
    TorchState,
)

# 检测 JAX 后端与 DLPack 零拷贝配对设备
JAX_GPU_DEVICES = [d for d in jax.devices() if d.platform == "gpu"]
JAX_PRIMARY_DEVICE = JAX_GPU_DEVICES[0] if JAX_GPU_DEVICES else jax.devices("cpu")[0]

if JAX_GPU_DEVICES and torch.cuda.is_available():
    DLPACK_JAX_DEVICE = JAX_GPU_DEVICES[0]
    DLPACK_TORCH_DEVICE = "cuda:0"
else:
    DLPACK_JAX_DEVICE = jax.devices("cpu")[0]
    DLPACK_TORCH_DEVICE = "cpu"

print(json.dumps({
    "pyrpmd_version": pyrpmd.__version__,
    "jax_version": jax.__version__,
    "jax_default_backend": jax.default_backend(),
    "jax_primary_device": str(JAX_PRIMARY_DEVICE),
    "dlpack_paired_devices": {
        "jax": str(DLPACK_JAX_DEVICE),
        "torch": DLPACK_TORCH_DEVICE,
    },
    "float64_enabled": bool(jnp.ones(1).dtype == jnp.float64),
}, indent=2))

## 1. 构建束缚态 $\mathrm{H_2O}$ JaxPIP 势能面并封装为 `PyRPMD` `Potential`

为了在 300 K 下稳定运行数百步量子环聚合物动力学，我们先在平衡水分子构型附近采样训练构型，利用三阶对称 PIP 基组拟合一个真实的束缚态非谐伸缩-弯曲教学水分子势能面（包含快变线性 PIP 基线 `water_linear_pes` 与非线性 PIP-NN 修正项 `water_nn_corr`），随后将其封装为遵循 `PyRPMD` 原子单位契约的 `JaxPIPPotential`，并用 `check_potential` 检验有限差分力精度。

In [ ]:
# 构造 H2O (原子顺序 ["H", "H", "O"]) 三阶置换不变多项式基组
# 核间距顺序: d0 = r(H1-H2), d1 = r(H1-O), d2 = r(H2-O)；交换 H1<->H2 即交换 d1<->d2
WATER_PIP_BASIS = [
    [[0, 0, 0]],
    [[0, 0, 1], [0, 1, 0]],
    [[1, 0, 0]],
    [[0, 1, 1]],
    [[1, 0, 1], [1, 1, 0]],
    [[0, 0, 2], [0, 2, 0]],
    [[2, 0, 0]],
    [[0, 1, 2], [0, 2, 1]],
    [[1, 1, 1]],
    [[0, 0, 3], [0, 3, 0]],
    [[3, 0, 0]],
]

OH_EQ_ANG = 0.9572
HOH_EQ_RAD = np.deg2rad(104.52)
WATER_EQ_ANG = np.array([
    [0.0, OH_EQ_ANG * np.sin(HOH_EQ_RAD / 2.0), OH_EQ_ANG * np.cos(HOH_EQ_RAD / 2.0)],
    [0.0, -OH_EQ_ANG * np.sin(HOH_EQ_RAD / 2.0), OH_EQ_ANG * np.cos(HOH_EQ_RAD / 2.0)],
    [0.0, 0.0, 0.0],
], dtype=np.float64)
HH_EQ_ANG = float(np.linalg.norm(WATER_EQ_ANG[0] - WATER_EQ_ANG[1]))


def analytic_water_target_ev(q_ang: jnp.ndarray) -> jnp.ndarray:
    '''平滑束缚态 H2O Morse 伸缩 + 谐性 H-H 耦合基准势 (单位: eV)，用于训练教学 PIP 模型。'''
    r_hh = jnp.linalg.norm(q_ang[0] - q_ang[1])
    r_oh1 = jnp.linalg.norm(q_ang[0] - q_ang[2])
    r_oh2 = jnp.linalg.norm(q_ang[1] - q_ang[2])
    de_ev, a_oh = 5.10, 2.15
    v_stretch = de_ev * ((1.0 - jnp.exp(-a_oh * (r_oh1 - OH_EQ_ANG))) ** 2
                         + (1.0 - jnp.exp(-a_oh * (r_oh2 - OH_EQ_ANG))) ** 2)
    v_bend = 2.40 * (r_hh - HH_EQ_ANG) ** 2 + 0.15 * (r_oh1 - OH_EQ_ANG) * (r_oh2 - OH_EQ_ANG)
    return v_stretch + v_bend


# 在平衡构型附近采样 48 个微扰构型拟合线性 PIP 模型 (eV / angstrom)
rng = np.random.default_rng(2026)
train_q_ang = jnp.asarray(WATER_EQ_ANG[None] + rng.normal(0.0, 0.05, size=(48, 3, 3)))
train_e_ev = jax.vmap(analytic_water_target_ev)(train_q_ang)

pip_desc = PolynomialDescriptor(WATER_PIP_BASIS, alpha=1.2, dtype=jnp.float64)
phi_mat = np.asarray(jax.vmap(pip_desc)(train_q_ang))
lin_coeffs, *_ = np.linalg.lstsq(phi_mat, np.asarray(train_e_ev), rcond=None)
water_linear_model = PolynomialLinearModel(pip_desc, coeffs=jnp.asarray(lin_coeffs))

# 构造一个温和的高阶 PIP-NN 修正项模型 (模拟 Δ-ML / 慢变高阶残差修正)
water_nn_corr = PolynomialNeuralNetwork(
    pip_desc, hidden_layers=[12, 12], key=jax.random.PRNGKey(17), activation="tanh"
)
water_nn_corr = water_nn_corr.update_scaler(
    jnp.asarray(phi_mat), jnp.linspace(-0.015, 0.015, len(train_q_ang))
)


class JaxPIPPotential(Potential):
    '''将 JaxPIP 模型通过 XLA filter_jit + vmap 封装为标准 PyRPMD 批量势能接口。'''

    def __init__(self, model, *, jax_device=JAX_PRIMARY_DEVICE):
        self.jax_device = jax_device
        self.metadata = {
            "provider": "jaxpip-xla",
            "execution_device": jax_device.platform,
            "batch_independent": True,
        }

        @eqx.filter_jit
        def _eval_batch(q_ang):
            return jax.vmap(model.get_energy_and_forces)(q_ang)

        self._eval_batch = _eval_batch

    def evaluate(self, q_bohr: np.ndarray) -> PESResult:
        q_ang = jax.device_put(jnp.asarray(q_bohr, dtype=jnp.float64) * BOHR_ANGSTROM, self.jax_device)
        e_ev, f_eva = jax.device_get(self._eval_batch(q_ang))
        return PESResult(
            energy=np.asarray(e_ev, dtype=np.float64) * EV_TO_AU,
            forces=np.asarray(f_eva, dtype=np.float64) * (EV_TO_AU * BOHR_ANGSTROM),
        )


jaxpip_pes = JaxPIPPotential(water_linear_model)
probe_bohr = np.asarray(train_q_ang[:4]) / BOHR_ANGSTROM
fd_check = check_potential(jaxpip_pes, probe_bohr, epsilon=1e-5)

print(json.dumps({
    "linear_fit_train_rmse_meV": float(np.sqrt(np.mean((phi_mat @ lin_coeffs - np.asarray(train_e_ev)) ** 2)) * 1e3),
    "provider_metadata": jaxpip_pes.metadata,
    "finite_difference_max_force_error_au": fd_check["max_absolute_force_error"],
}, indent=2))

assert fd_check["max_absolute_force_error"] < 1e-8

## 2. DLPack 零拷贝桥接接入 `PyRPMD` 设备驻留张量引擎 (`TorchPES` + `TorchDynamics` + RPC/MTS)

当同时使用 `PyRPMD` 的设备驻留积分器 `TorchDynamics`（提供 `cayley-bcocb` 主平方根分裂、`pile-l`/`trpmd` 恒温器、环聚合物收缩 `TorchRPCTerm` 与多时间步长 `TorchSplitPotential`）与 `JaxPIP` 势能面时，可通过 **DLPack 协议**直接传递底层缓冲区指针：
- `jnp.from_dlpack(q_torch.detach().contiguous())` 将 `[B*P, N, 3]` 设备张量直接暴露给 XLA 编译后的 `JaxPIP` 内核；
- 求值得到的 `(energy, forces)` 再经 `torch.from_dlpack(...)` 直接返回给 `TorchPES`（由于已显式返回解析力，`TorchPES` 自动跳过 `torch.autograd`，无额外图开销）。

In [ ]:
class JaxPIPTorchBridge(torch.nn.Module):
    '''通过 DLPack 零拷贝将 JaxPIP 模型桥接为 TorchPES 可直接调用的批量模块。'''

    metadata = {"batch_independent": True}

    def __init__(self, jax_model):
        super().__init__()

        @eqx.filter_jit
        def _batch_energy_forces(q_model):
            return jax.vmap(jax_model.get_energy_and_forces)(q_model)

        self._batch_energy_forces = _batch_energy_forces

    def forward(self, q_model: torch.Tensor):
        q_contig = q_model.detach().contiguous()
        q_jax = jnp.from_dlpack(q_contig)
        e_jax, f_jax = self._batch_energy_forces(q_jax)
        return torch.from_dlpack(e_jax), torch.from_dlpack(f_jax)


# 2.1 验证 DLPack 零拷贝共享内存地址与单位换算一致性
sample_t = torch.randn((4, 3, 3), dtype=torch.float64, device=DLPACK_TORCH_DEVICE)
roundtrip_t = torch.from_dlpack(jnp.from_dlpack(sample_t))
zero_copy_verified = bool(sample_t.data_ptr() == roundtrip_t.data_ptr())

# length_factor 为每模型长度单位对应的 bohr 数 (1 angstrom = 1 / BOHR_ANGSTROM bohr)
fast_torch_pes = TorchPES(
    JaxPIPTorchBridge(water_linear_model),
    device=DLPACK_TORCH_DEVICE,
    length_factor=1.0 / BOHR_ANGSTROM,
    energy_factor=EV_TO_AU,
)
slow_corr_pes = TorchPES(
    JaxPIPTorchBridge(water_nn_corr),
    device=DLPACK_TORCH_DEVICE,
    length_factor=1.0 / BOHR_ANGSTROM,
    energy_factor=EV_TO_AU,
)

# 核对 TorchPES(JaxPIPTorchBridge) 与第 1 节 JaxPIPPotential 的能量和力完全一致
ref_res = jaxpip_pes.evaluate(probe_bohr)
bridge_res = fast_torch_pes.evaluate_tensor(
    torch.as_tensor(probe_bohr, dtype=torch.float64, device=fast_torch_pes.device)
)
np.testing.assert_allclose(bridge_res.energy.detach().cpu().numpy(), ref_res.energy, atol=1e-13, rtol=1e-12)
np.testing.assert_allclose(bridge_res.forces.detach().cpu().numpy(), ref_res.forces, atol=1e-13, rtol=1e-12)

# 2.2 组合环聚合物收缩 (RPC: P=16 -> Pc=4) 与多时间步长 (MTS: inner_steps=2) 运行批量 H2O PIMD
n_traj, n_beads = 8, 16
rpc_split_pes = TorchSplitPotential(
    fast=[TorchRPCTerm(fast_torch_pes, nbeads=n_beads, contracted_beads=n_beads)],
    slow=[TorchRPCTerm(slow_corr_pes, nbeads=n_beads, contracted_beads=4)],
)

water_masses_au = np.array([1.00782503223, 1.00782503223, 15.99491461957]) * AMU_TO_AU
beta_300k = 1.0 / (300.0 * 3.166811563e-6)

rpc_dyn = TorchDynamics(
    rpc_split_pes,
    masses=water_masses_au,
    beta=beta_300k,
    nbeads=n_beads,
    dt=8.0,  # 外步长 ~0.19 fs，内步长 h = 4.0 a.u. (~0.097 fs)
    inner_steps=2,
    integrator="cayley-bcocb",
    thermostat="pile-l",
    gamma_centroid=0.002,
    seed=2026,
)

q0_bohr = np.broadcast_to((WATER_EQ_ANG / BOHR_ANGSTROM)[None, None], (n_traj, n_beads, 3, 3)).copy()
rpc_state = TorchState.from_arrays(q0_bohr, rpc_dyn.sample_momenta(n_traj), device=fast_torch_pes.device)
rpc_dyn.propagate(rpc_state, 40)

q_snap = rpc_state.snapshot()["q"]
centroid = q_snap.mean(axis=1, keepdims=True)
rg_h_bohr = float(np.sqrt(np.mean(np.sum((q_snap[:, :, :2, :] - centroid[:, :, :2, :]) ** 2, axis=-1))))
rg_o_bohr = float(np.sqrt(np.mean(np.sum((q_snap[:, :, 2:, :] - centroid[:, :, 2:, :]) ** 2, axis=-1))))

print(json.dumps({
    "dlpack_zero_copy_shared_ptr": zero_copy_verified,
    "state_shape": list(rpc_state.q.shape),
    "force_evaluations": rpc_dyn.force_evaluations,
    "hydrogen_rms_rg_bohr": round(rg_h_bohr, 5),
    "oxygen_rms_rg_bohr": round(rg_o_bohr, 5),
}, indent=2))

assert zero_copy_verified is True
assert rpc_dyn.force_evaluations["fast"] > rpc_dyn.force_evaluations["slow"]
assert rg_h_bohr > rg_o_bohr  # 轻核 H 的量子离域半径显著大于重核 O

## 3. 纯 JAX XLA 全编译融合传播循环 (`jax.lax.scan`) 与 `PyRPMD` 精度对标

若希望将**多步环聚合物积分循环本身**也与 `JaxPIP` 势能求值完全融合进同一个 XLA 编译内核（在 GPU 上实现连续数十至数百步零主机交互执行），可使用 `jax.lax.scan` 编写与 `PyRPMD` 数学定义完全一致的法向模 Cayley 分裂 `B(dt/2) -> C(dt) -> B(dt/2)`。

下面我们对比 60 步双轨迹、8 珠子 $\mathrm{H_2O}$ NVE 环聚合物动力学在：
1. `PyRPMD` 的 `TorchDynamics(..., integrator="cayley-bcb", thermostat=None)`；
2. 纯 JAX XLA 全图编译 `jax_propagate_cayley_scan`

两条路径上的末态坐标/动量逐分量最大偏差以及环聚合物总哈密顿量守恒精度。

In [ ]:
@eqx.filter_jit
def jax_propagate_cayley_scan(model, q_init, p_init, masses_au, beta_au, dt_au, nsteps: int):
    '''在单个 XLA 编译图内融合 nsteps 步环聚合物 Cayley 传播与 JaxPIP 批量力求值。'''
    b_sz, p_cnt, n_at, _ = q_init.shape
    m = masses_au.reshape(1, 1, n_at, 1)
    k_idx = jnp.arange(p_cnt // 2 + 1, dtype=jnp.float64)
    omega = 2.0 * p_cnt / beta_au * jnp.sin(jnp.pi * k_idx / p_cnt)
    z = 0.5 * dt_au * omega
    den = 1.0 + z * z
    ca = ((1.0 - z * z) / den).reshape(1, -1, 1, 1)
    cb = (dt_au / den).reshape(1, -1, 1, 1)
    cc = (dt_au * omega * omega / den).reshape(1, -1, 1, 1)

    def eval_batch_au(q_ring):
        flat_ang = q_ring.reshape(b_sz * p_cnt, n_at, 3) * BOHR_ANGSTROM
        e_ev, f_eva = jax.vmap(model.get_energy_and_forces)(flat_ang)
        return (
            e_ev.reshape(b_sz, p_cnt) * EV_TO_AU,
            f_eva.reshape(b_sz, p_cnt, n_at, 3) * (EV_TO_AU * BOHR_ANGSTROM),
        )

    _, f0 = eval_batch_au(q_init)

    def step_fn(carry, _):
        q, p, f = carry
        p = p + 0.5 * dt_au * f
        qk = jnp.fft.rfft(q, axis=1, norm="ortho")
        pk = jnp.fft.rfft(p, axis=1, norm="ortho")
        q = jnp.fft.irfft(ca * qk + cb * pk / m, n=p_cnt, axis=1, norm="ortho")
        p = jnp.fft.irfft(ca * pk - cc * m * qk, n=p_cnt, axis=1, norm="ortho")
        e_pot, f_new = eval_batch_au(q)
        p = p + 0.5 * dt_au * f_new

        kinetic = jnp.sum(p * p / (2.0 * m), axis=(1, 2, 3))
        dq = q - jnp.roll(q, -1, axis=1)
        spring = 0.5 * (p_cnt / beta_au) ** 2 * jnp.sum(m * dq * dq, axis=(1, 2, 3))
        h_ring = jnp.sum(e_pot, axis=1) + kinetic + spring
        return (q, p, f_new), h_ring

    (q_final, p_final, _), h_history = jax.lax.scan(step_fn, (q_init, p_init, f0), None, length=nsteps)
    return q_final, p_final, h_history


beads_cmp, steps_cmp, dt_cmp = 8, 60, 4.0
q_init_cmp = (
    np.broadcast_to((WATER_EQ_ANG / BOHR_ANGSTROM)[None, None], (2, beads_cmp, 3, 3))
    + rng.normal(0.0, 0.015, size=(2, beads_cmp, 3, 3))
)
p_init_cmp = rng.normal(0.0, 0.4, size=(2, beads_cmp, 3, 3))

# 路径 A: PyRPMD TorchDynamics (cayley-bcb)
ref_dyn = TorchDynamics(
    fast_torch_pes,
    masses=water_masses_au,
    beta=beta_300k,
    nbeads=beads_cmp,
    dt=dt_cmp,
    integrator="cayley-bcb",
    thermostat=None,
)
ref_state = TorchState.from_arrays(q_init_cmp, p_init_cmp, device=fast_torch_pes.device)
ref_dyn.propagate(ref_state, steps_cmp)
ref_snap = ref_state.snapshot()

# 路径 B: 纯 JAX XLA lax.scan 融合传播 (在 JAX_PRIMARY_DEVICE 上执行)
q_jax_in = jax.device_put(jnp.asarray(q_init_cmp), JAX_PRIMARY_DEVICE)
p_jax_in = jax.device_put(jnp.asarray(p_init_cmp), JAX_PRIMARY_DEVICE)
m_jax_in = jax.device_put(jnp.asarray(water_masses_au), JAX_PRIMARY_DEVICE)

qf_jax, pf_jax, h_hist_jax = jax_propagate_cayley_scan(
    water_linear_model, q_jax_in, p_jax_in, m_jax_in, beta_300k, dt_cmp, steps_cmp
)
qf_np, pf_np, h_hist_np = map(np.asarray, jax.device_get((qf_jax, pf_jax, h_hist_jax)))

max_q_diff = float(np.max(np.abs(qf_np - ref_snap["q"])))
max_p_diff = float(np.max(np.abs(pf_np - ref_snap["p"])))
max_h_drift = float(np.max(np.abs(h_hist_np - h_hist_np[0:1, :])))

print(json.dumps({
    "jax_xla_device": str(JAX_PRIMARY_DEVICE),
    "steps": steps_cmp,
    "max_abs_q_diff_vs_pyrpmd_bohr": max_q_diff,
    "max_abs_p_diff_vs_pyrpmd_au": max_p_diff,
    "max_per_trajectory_hamiltonian_drift_hartree": max_h_drift,
}, indent=2))

assert max_q_diff < 1e-12
assert max_p_diff < 1e-11
assert max_h_drift < 1e-4

## 4. 大规模环聚合物批量吞吐量扩展评测 (`B * P` XLA Batch Scaling)

JaxPIP 的稠密张量化设计使单次调用能够同时处理成百上千个分子珠子构型（如 $B = 64$ 条轨迹 $\times$ $P = 16$ 个珠子 $= 1024$ 个水分子构型/步）。下面我们在预热 XLA 编译缓存后，使用 `jax.block_until_ready()` 同步设备计时，测量不同轨迹批量大小 $B \in \{1, 8, 32, 64\}$ 下融合 `lax.scan` 动力学的总吞吐量（`bead_steps_per_second`）与单珠子单步摊销微秒耗时。

In [ ]:
bench_beads = 16
bench_steps = 50
m_dev = jax.device_put(jnp.asarray(water_masses_au), JAX_PRIMARY_DEVICE)

scaling_results = []
for batch_traj in (1, 8, 32, 64):
    q_b = jax.device_put(
        jnp.broadcast_to(
            jnp.asarray(WATER_EQ_ANG / BOHR_ANGSTROM)[None, None],
            (batch_traj, bench_beads, 3, 3),
        ),
        JAX_PRIMARY_DEVICE,
    )
    p_b = jax.device_put(
        jnp.asarray(rng.normal(0.0, 0.3, size=(batch_traj, bench_beads, 3, 3))),
        JAX_PRIMARY_DEVICE,
    )

    # 预热当前形状的 XLA 编译产物，将 JIT 编译耗时与稳态传播耗时严格分离
    warm_q, _, _ = jax_propagate_cayley_scan(
        water_nn_corr, q_b, p_b, m_dev, beta_300k, 2.0, bench_steps
    )
    jax.block_until_ready(warm_q)

    t0 = time.perf_counter()
    out_q, _, _ = jax_propagate_cayley_scan(
        water_nn_corr, q_b, p_b, m_dev, beta_300k, 2.0, bench_steps
    )
    jax.block_until_ready(out_q)
    elapsed = time.perf_counter() - t0

    total_bead_steps = batch_traj * bench_beads * bench_steps
    scaling_results.append({
        "batch_trajectories": batch_traj,
        "concurrent_bead_frames": batch_traj * bench_beads,
        "wall_seconds": round(elapsed, 4),
        "bead_steps_per_second": round(total_bead_steps / elapsed, 1),
        "amortized_us_per_bead_step": round(elapsed * 1e6 / total_bead_steps, 3),
    })

print(json.dumps({
    "backend": jax.default_backend(),
    "device": str(JAX_PRIMARY_DEVICE),
    "model": "H2O PolynomialNeuralNetwork (hidden=[12, 12], float64)",
    "beads_per_trajectory": bench_beads,
    "steps_per_launch": bench_steps,
    "scaling": scaling_results,
}, indent=2))

assert scaling_results[-1]["bead_steps_per_second"] > scaling_results[0]["bead_steps_per_second"]